In [1]:
import pandas as pd
import time
from thor_requests.connect import Connect
from thor_requests.wallet import Wallet
from thor_requests.contract import Contract
import json
import os


In [2]:
# config 
RPC = "https://vethor-node-test.vechaindev.com" #testnet
#RPC = "" #mainnet
connector = Connect(RPC)
SMC_VESTING_ADDRESS = os.getenv('iVBAirdrop')

key_dict = {}
with open("../../keystore") as json_file:
  key_dict = json.load(json_file)
_owner = "0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7"
_wallet = Wallet.fromKeyStore(ks=key_dict, password='passtest')

_contract = Contract.fromFile('../abi/VBAirdrop.json')

In [3]:
def check_beneficiary(address):
  try:
    res = connector.call(
    caller=_owner,
    contract=_contract, 
    func_name="getBeneficiary", 
    func_params=[address],
    to=SMC_VESTING_ADDRESS,
    )
    return res
  except:
    return None
  return None
    
# aaa = check_beneficiary("0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB")
# print(aaa)
def add_beneficiary(address, amount):
  txn = connector.transact(
    wallet=_wallet,
    contract=_contract,
    func_name="addBeneficiary",
    func_params=[address,amount],
    to=SMC_VESTING_ADDRESS,
    
  )
  id = txn["id"]
  tx_id = connector.wait_for_tx_receipt(tx_id=id, timeout=20)
  return id,tx_id

# print(add_beneficiary("0x9a773a0c1710a5afd9d25eb5b0d2dca2239663e6",1300000000))
# print(connector.get_tx("0x314f4fbf3e680f01e594e72704cee7c5f0a88ce89420143a968cbc4e832a7609"))

def is_confirmed(tx_hash):
    try:
        receipt = connector.get_tx_receipt(tx_id=tx_hash)
        if str(receipt["reverted"]) == "False": ## transaction success => receipt["reverted"] == False  
            return True
        return False
    except:
        return False

# print(is_confirmed("0x9fedba3fab7b5953978f606ab6e6ca5e679e5c4683430279a1c93a6ee1d360e6"))


In [4]:
progress = []


In [5]:
# added = []

# Input the data file in csv format
data = pd.read_csv("data.csv")
for row in data.iterrows():
  address = row[1][0]
  amount = int(row[1][1])*10**18
  if amount <=0: continue
  check = check_beneficiary(address)
  if check is not None:
    if str(check['reverted']) == "False":
      print(f"beneficiary {address} is already existed, skip")
      # added.append({"address": address, "amount": amount, "status": "skip"})
      continue
  tx_id,tx_hash = add_beneficiary(address, amount)
  if tx_hash != "":
    print(f"adding beneficiary {address} , txhash: {tx_hash}")
    progress.append({"address": address, "amount": amount, "status": "added", "hash": tx_id})
  time.sleep(5)

beneficiary 0xE4A482E15Bd8D5cAEf13B2f0EfdE7Bf15B737929 is already existed, skip
beneficiary 0xaDF66a56f668Bd18C598af93207330273E62ccA8 is already existed, skip
beneficiary 0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB is already existed, skip
beneficiary 0x817f3fb962b5b090356e953134f34e63c5a7a0ad is already existed, skip
beneficiary 0xfaae2dddb4afc844533f214273c0d89819d728e0 is already existed, skip
adding beneficiary 0xd4601ccac345484e84f8fed6fb02a40d13108c61 , txhash: {'gasUsed': 114844, 'gasPayer': '0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7', 'paid': '0xff0141865d98000', 'reward': '0x4c8060751c14000', 'reverted': False, 'meta': {'blockID': '0x00c57b5873f609aaf0a4207242b3e447c807046c789db5e4b3597d0730baf866', 'blockNumber': 12942168, 'blockTimestamp': 1659453030, 'txID': '0xd39c4553c41bfe0d0f4c9a653ea14c1d91d5b41fcc135587625a9ba6c8e1f2ad', 'txOrigin': '0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7'}, 'outputs': [{'contractAddress': None, 'events': [{'address': '0xc652898b0b05bbe0650ad4ff0

In [7]:
final = []
for item in progress:
  if "hash" in item:
    check = is_confirmed(item["hash"])
    item["confirm"] = check
    final.append(item)
df = pd.DataFrame(final)
df.to_csv('../AirdropVesting/save.csv')
df

,address,amount,status,hash,confirm
0,0xd4601ccac345484e84f8fed6fb02a40d13108c61,1006000000000000000000,added,0xd39c4553c41bfe0d0f4c9a653ea14c1d91d5b41fcc13...,True
1,0x27b6b7245f0df46f5ce35658b2aeeb97d0f64912,1007000000000000000000,added,0xc8d87fc6ba3f47ef0ca33f121fc67347016d77ac14a2...,True
2,0x8c330dc0f8b4039a4b4a9f386b3095f49d1dafa9,1008000000000000000000,added,0x00ee1152c027852a198452c6c6606e0400f9483ffcf6...,True
3,0x4dabdfc1d1304e99be2a5ecdb62b96ca58bdc16a,1009000000000000000000,added,0x1ee32a628365b266137f200cce7e483c467a16724c05...,True
4,0xb60b5208684ec09288f5e36b0d3caeceee343c82,1010000000000000000000,added,0x241bda2db79394f9b5304a6c69a3d24bf4902220d8bd...,True
